# 基于电影评分的协同过滤推荐系统 UserCF (基于记忆的协同过滤和基于模型的协同过滤)
# Collaborative Filtering Recommendation System Based on Movie Ratings (Memory-based CF and Model-based CF) V0.4
**GDUFE cclear116 for self-study 2025/3/29-2025/4/3**

**更新日志**
1. 调整了笔记的整体目录，增加了基于模型的协同过滤部分 2025/4/6-23:00
2. 改正了UserCF中最后选取推荐哪些电影的代码，修复完善了加权平均分计算部分 2025/4/6-23:48
3. 根据AccidieMiu提交的issue，修改了ItemCF部分的总结 2025/4/6-23:52
4. 他妈的，改进的代码有逻辑上的bug，重做了 2025/4/7-00:30

# Part1-基于记忆的协同过滤
---

# 1 准备

## 1.1 数据来源
 Link:[MovieLens](https://grouplens.org/datasets/movielens/)

[**Summary**]

This dataset (ml-latest-small) describes 5-star rating and free-text tagging activity from [MovieLens](http://movielens.org), a movie recommendation service. It contains 100836 ratings and 3683 tag applications across 9742 movies. These data were created by 610 users between March 29, 1996 and September 24, 2018. This dataset was generated on September 26, 2018.

Users were selected at random for inclusion. All selected users had rated at least 20 movies. No demographic information is included. Each user is represented by an id, and no other information is provided.

The data are contained in the files `links.csv`, `movies.csv`, `ratings.csv` and `tags.csv`. More details about the contents and use of all these files follows.

This is a *development* dataset. As such, it may change over time and is not an appropriate dataset for shared research results. See available *benchmark* datasets if that is your intent.

This and other GroupLens data sets are publicly available for download at <http://grouplens.org/datasets/>.

[**Reference**]

[1]. [推荐系统实践--基于用户的协同过滤算法](https://www.cnblogs.com/qwj-sysu/p/4368874.html)

[2]. [协同过滤算法介绍及算法实现](https://www.cnblogs.com/1113127139aaa/p/9830449.html)

[3]. [最全面的推荐系统评估方法介绍](https://blog.csdn.net/m0_37586850/article/details/109664623)

[4]. [recommenderlab: A Framework for Developing and Testing Recommendation Algorithms](https://www.researchgate.net/publication/237246291_recommenderlab_A_Framework_for_Developing_and_Testing_Recommendation_Algorithms)

## 1.2 一些基本知识
大部分的推荐系统其工作原理还是基于物品或用户的相似性进行推荐

(1)以下是一些典型的推荐方法分类：
- 基于人口统计学的推荐(Demographic-based Recommendation)
- 基于内容的推荐(Content-Based Recommendation)
- 基于协同过滤的推荐 (Collaborative Filtering-Based Recommendation) **（这次主要探讨这个）**
- 混合型推荐系统 (Hybrid Recommendation)

(2) 实现协同过滤，我们需要遵循以下步骤：
1. 收集用户偏好
2. 识别相似用户或物品
3. 计算推荐
    
(3) 相似度计算
1. 皮尔逊相关系数 **(√)**
    - 适用于ItemCF和UserCF
    - 能够捕捉评分模式的相似性，能够更好地反映用户的评分偏好。但计算复杂度比较高。
2. 余弦相似度
    - 适用于UserCF，特别是在处理高维稀疏数据时
    - 计算效率高，但忽略评分的大小差异。对评分的绝对值差异不敏感，可能无法捕捉评分模式的细微变化。
3. 调整余弦相似度
    - 同上，但考虑了用户评分的平均值，能够消除用户评分偏好的影响。在处理评分偏差较大的数据时表现更好。
    - 但计算复杂度比较高。
4. 欧几里得距离 与 曼哈顿距离
    - 易于理解计算简单，但效果不好
5. 杰卡德相似度
    - 适用于基于物品的协同过滤，特别是在评分数据中存在大量未评分的电影时。
    - 适用于二元数据（如用户是否观看过某部电影）。无法处理连续的评分数据。
6. 模型算法 高成本高回报/其他算法略

## 1.3 关于此次使用的协同过滤算法

协同过滤是推荐系统中常用的技术，基于用户对物品的评分数据来预测缺失评分或生成推荐列表，主要分为基于内存的协同过滤和基于模型的协同过滤算法。
 1. **基于用户的协同过滤（√）**
    - 通过分析相似用户的评分来预测目标用户的缺失评分。利用皮尔逊相关系数、余弦相似度等计算用户间相似度，选取相似用户形成邻域，聚合邻域用户评分得到预测评分。
    - 还可通过归一化处理消除用户评分偏差，以提升预测准确性。但该方法存在内存和计算效率问题，因为需要存储整个用户数据库，且计算相似度时计算量较大。
 2. **基于物品的协同过滤（？）**
    - 基于物品间的相似关系进行推荐。计算物品间相似度构建相似矩阵，选取k个最相似物品，根据用户对相关物品的评分计算加权和得到预测评分。
    - 相比基于用户的协同过滤，其效率更高，在大规模推荐系统中应用广泛，原因是物品相似度矩阵相对稳定且规模较小，可预先计算存储。
 3. **基于0 - 1数据的协同过滤** (不适合数据，不讨论)
    - 在缺乏详细评分数据时，通过分析用户行为得到0 - 1数据。处理此类数据时可假设缺失值为负例或未知，还可使用Jaccard指数计算相似度，以衡量用户或物品之间的相似程度，进而进行推荐。
    - **对于此次的电影评分数据并不适用。如果使用这种方法，会使得原本有价值的评分数据得不到有效利用从而使预测效果降低。**
 4. **基于关联规则的推荐（√）** (基于模型的算法)
    - 将用户对物品的偏好数据视为交易数据，挖掘关联规则构建依赖模型。根据用户喜欢的物品和关联规则，推荐匹配规则中置信度高的物品。
    - 常用支持度和置信度等指标来衡量关联规则的有效性，支持度表示规则在数据集中出现的频率，置信度表示在满足前件的情况下出现后件的概率。
 5. 其他协同过滤算法

    省略...

## 1.4 推荐算法的评估
通过将用户划分为训练集和测试集，用训练集学习模型，测试集评估模型预测未知物品评分或推荐列表的能力。
 1. **预测评分的评估**
    - 常用平均绝对误差（MAE）和均方根误差（RMSE）衡量预测评分与真实评分的偏差。MAE计算预测评分与真实评分差值的绝对值的平均值，直观反映预测的平均误差程度；
    - RMSE对预测评分与真实评分差值的平方和求平均再开方，对较大误差的惩罚更大，更能体现预测评分的整体偏差情况。
 2. **Top - N推荐的评估**
    - 将预测的Top - N推荐列表与测试集中用户喜欢的物品进行比较，构建混淆矩阵。从混淆矩阵中可衍生出准确率、召回率、F - measure等性能指标，用于评估推荐效果。
    - 还可使用ROC曲线评估算法性能，ROC曲线以真正例率（等同于召回率）为纵坐标，假正例率为横坐标，曲线下的面积（AUC）越大，算法性能越好。 

# 2.数据预处理
## 2.1 导入数据/导入所需库

In [92]:
import pandas as pd
import numpy as np
import time

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

## 2.2数据预览

In [95]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [96]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [99]:
print('Data info:')
print('\nmovies:',movies.info())
print('\nratings:',ratings.info())

Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB

movies: None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB

ratings: None


可以看到数据无缺失值

## 2.2检测是否有重复值（重复评分）

In [102]:
duplicate_rows = ratings.duplicated(subset=['userId','movieId'],keep=False)
ratings[duplicate_rows]

,userId,movieId,rating,timestamp


**可见没有重复评分，无需进行后续处理**

如果有重复值，可以这样操作：
```python
# 按照 userId 和 movieId 分组，只保留最新的评论
ratings = ratings.sort_values(by='timestamp', ascending=False).groupby(['userId', 'movieId']).first().reset_index()
```

# 3.构建推荐系统
## 3.1 创建 评分矩阵 和 映射
- **ratings.pivot**：将 ratings 数据框转换为一个矩阵，其中行索引是 userId，列索引是 movieId，矩阵中的值是对应的评分。以此构建一个以用户为行、电影为列，矩阵元素为用户对电影评分的二维表格，方便后续计算用户之间的相似度。
- **过滤电影**：通过 isin 方法筛选出 movies 数据框中 movieId 存在于 ratings_df 列中的电影，即只保留那些有评分记录的电影。将筛选后的 valid_movies 数据框的 movieId 列设置为索引，然后提取 title 列，最后将其转换为字典。这样方便后续根据电影 ID 快速查找对应的电影名称。

In [105]:
# 创建评分矩阵
ratings_df = ratings.pivot(index='userId', columns='movieId', values='rating')

# 创建电影ID到名称的映射（自动过滤不存在于评分矩阵的电影）
valid_movies = movies[movies['movieId'].isin(ratings_df.columns)]
movie_id_name = valid_movies.set_index('movieId')['title'].to_dict()

In [106]:
ratings_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3.2计算用户相似度
- **ratings_df.T**：对评分矩阵进行转置，将行和列交换，这样就可以计算用户之间的相似度。因为 corr 方法默认是计算列之间的相关性，转置后就可以计算用户之间的相关性。
- **corr**(method='pearson', min_periods=5)：使用皮尔逊相关系数来计算用户之间的相似度。min_periods=5 表示两个用户之间至少要有 5 个共同评分的电影，才能计算它们之间的相似度，避免因共同评分电影过少导致相似度计算不准确。

In [108]:
user_similarity = ratings_df.T.corr(method='pearson', min_periods=5)  # 至少需要5个共同评分项

## 3.3创建用户ID到索引的映射
- **快速定位**：使用字典推导式创建一个字典 user_id_map，将用户 ID 映射到评分矩阵中的行索引。这样方便后续根据用户 ID 快速定位到评分矩阵中的对应行。

In [114]:
# 用户ID到索引的映射（使用评分矩阵的索引）
user_id_map = {user_id: idx for idx, user_id in enumerate(ratings_df.index)}

#### 3.4推荐生成函数
1. 冷启动处理：如果目标用户 ID 不在 user_id_map 中，说明该用户不存在或未评分任何电影。此时，使用 value_counts 方法**统计每个电影的评分次数，然后取评分次数最多的前 top_n 部电影作为推荐结果**。
2. 寻找相似用户：根据目标用户的索引，从相似度矩阵中获取该用户与其他用户的相似度得分，筛选出相似度大于 0.2 且不是目标用户自己的用户，取相似度最高的前 50 个用户。
3. 收集候选电影：遍历相似用户，找出他们评分过但目标用户未评分的电影，将这些电影添加到候选电影集合中。
4. 处理无候选电影情况：如果候选电影集合为空，再次返回评分次数最多的前 top_n 部电影。
5. 加权推荐：对于候选电影，计算每个候选电影的加权评分然后推荐加权平均分数最高的前 top_n 部电影。

In [147]:
def generate_recommendations(target_user_id, top_n=5):
    
    # -----冷启动：用户不存在或未评分任何电影---------
    if target_user_id not in user_id_map:
        popular_movies = ratings['movieId'].value_counts().index[:top_n]
        # 字典的get方法，根据电影ID获取电影名称。如果找不到就返回“未知电影 (ID: xxx)”。
        return [movie_id_name.get(mid, f"未知电影 (ID: {mid})") for mid in popular_movies]

    # 根据目标用户 ID 从 user_id_map 中获取对应的行索引。
    user_idx = user_id_map[target_user_id]
    # 根据行索引获取目标用户的评分行，然后使用notna方法判断哪些电影有评分
    # 返回一个布尔数组 target_rated，即目标用户已经评分的电影。
    target_rated = ratings_df.iloc[user_idx].notna()

    # -----寻找有效相似用户（相似度>0.2且非自己）-------
    sim_scores = user_similarity.iloc[user_idx].values
    # 创建一个布尔掩码，筛选出相似度大于 0.2 且不是目标用户自己的用户
    valid_mask = (sim_scores > 0.2) & (np.arange(len(sim_scores)) != user_idx)
    # 降序排序，取前 50 个与目标用户最相似的用户
    similar_users = np.argsort(-sim_scores[valid_mask])[:50] 
    
    # ------收集候选电影（使用集合提高效率）-------
    candidate_movies = set() # 储存候选电影
    for sim_idx in similar_users:
        # 获取每个相似用户已经评分的电影，返回一个布尔数组
        sim_rated = ratings_df.iloc[sim_idx].notna()
        # 找出相似用户评分过但目标用户未评分的电影。
        new_movies = ratings_df.columns[sim_rated & ~target_rated]
        # 将这些新电影添加到候选电影集合中
        candidate_movies.update(new_movies)
    
    # -----处理无候选电影情况----- 
    # 如果候选电影集合为空，说明没有找到合适的推荐电影。此时，再次返回评分次数最多的前 top_n 部电影作为推荐结果。
    if not candidate_movies:
        popular_movies = ratings['movieId'].value_counts().index[:top_n]
        return [movie_id_name.get(mid, f"未知电影 (ID: {mid})") for mid in popular_movies]
    
    # -----加权推荐（考虑相似度和评分次数）-----
    # 将集合转换为列表 [*因为 Pandas 数据框的索引需要使用列表。*]
    candidate_movies = list(candidate_movies)
    # 统计每个候选电影的评分次数，并按照评分次数降序排序。
    movie_counts = ratings_df[candidate_movies].count().sort_values(ascending=False)

     # 计算每个候选电影的加权评分
    weighted_scores = {}
    for mid in candidate_movies:
        # 获取目标用户与所有相似用户的评分
        user_scores = ratings_df[mid].dropna()
        
        # 初始化加权评分和相似度总和
        weighted_score = 0.0  # 这里初始化加权平均分
        sim_sum = 0.0  # 初始化相似度总和
        
        for sim_idx in similar_users:
            sim_user_id = ratings_df.index[sim_idx]  # 获取相似用户的 ID
            if mid in ratings_df.columns and sim_user_id in user_id_map:
                sim_user_idx = user_id_map[sim_user_id]  # 获取相似用户的索引
                if not np.isnan(ratings_df.at[sim_user_id, mid]):  # 检查相似用户对当前电影是否有评分
                    # 这里计算：加权平均分 = 相似度 * 评分
                    weighted_score += sim_scores[sim_idx] * ratings_df.at[sim_user_id, mid]
                    sim_sum += sim_scores[sim_idx]  # 累加相似度
        
        # 平均加权评分 ：加权评分 / 相似用户的相似度总和
        if sim_sum > 0:
            weighted_scores[mid] = weighted_score / sim_sum
        else:
            weighted_scores[mid] = 0.0  # 如果相似度总和为零，设置为 0.0

    # 按加权评分降序排序
    sorted_movies = sorted(weighted_scores.items(), key=lambda x: x[1], reverse=True)
    
    #【测试输出】========================================================
    # print('\n target_rated即目标用户已经评分的电影:',target_rated) ####测试输出
    # print('\n valid_mask相似度大于 0.2的布尔掩码:',valid_mask) ####测试输出
    # print('\n  similar_users前 50 个与目标用户最相似的用户:',similar_users) ####测试输出
    # print('\n 候选电影的数量:',len(candidate_movies) ) ####测试输出
    # print('\n 候选电影的评分次数降序排列:',movie_counts ) ####测试输出
    # 格式化输出 weighted_scores
    # formatted_scores = {mid: score for mid, score in sorted_movies if not np.isnan(score)}
    # print(' weighted_scores:', formatted_scores)
    
    # 取评分次数最多的前 top_n 部电影作为推荐结果，并将电影 ID 转换为电影名称。
    return [movie_id_name.get(mid, f"未知电影 (ID: {mid})") for mid, _ in sorted_movies[:top_n]]

## 3.4执行流程

In [150]:
if __name__ == "__main__":
    # 计时
    start_time = time.time()
    
    # 定义一个目标用户列表，包含要进行推荐的用户 ID
    target_users = [1, 15, 99999]
    
    # 统一使用user_id_map进行存在性验证
    print("用户存在性验证：")
    # 遍历目标用户列表，检查每个用户 ID 是否存在于 user_id_map 中，并打印验证结果
    for uid in target_users:
        exists = "存在" if uid in user_id_map else "不存在"
        print(f"  用户 {uid}: {exists}")
    
    # 统一推荐接口
    for user_id in target_users:
        print(f"\n=== 用户 {user_id} 的推荐 ===")
        try:
            recs = generate_recommendations(user_id)
            for i, title in enumerate(recs, 1):
                print(f"{i}. {title}")
        except Exception as e:
            print(f"错误：{str(e)}")

    # 数据验证
    # 通过集合的差集操作，找出存在于评分矩阵中但没有电影名称的电影。
    missing_movies = set(ratings_df.columns) - set(movie_id_name.keys())
    print(f"\n存在于评分数据但缺失名称的电影数量: {len(missing_movies)}")

    #计时
    end_time = time.time()
    print(f"函数运行时间: {end_time - start_time} 秒")

用户存在性验证：
  用户 1: 存在
  用户 15: 存在
  用户 99999: 不存在

=== 用户 1 的推荐 ===
1. Babysitter, The (1995)
2. Next Karate Kid, The (1994)
3. RoboCop 3 (1993)
4. Scarface (1983)
5. Wolf (1994)

=== 用户 15 的推荐 ===
1. Raising Arizona (1987)
2. Bridget Jones's Diary (2001)
3. Truman Show, The (1998)
4. Dictator, The (2012)
5. Julie & Julia (2009)

=== 用户 99999 的推荐 ===
1. Forrest Gump (1994)
2. Shawshank Redemption, The (1994)
3. Pulp Fiction (1994)
4. Silence of the Lambs, The (1991)
5. Matrix, The (1999)

存在于评分数据但缺失名称的电影数量: 0
函数运行时间: 1.4635069370269775 秒


# 4.改进
### **1. 相似度计算优化（有点麻烦，运算远不如corr()高效）**
- **加入流行度惩罚因子**  
  修改相似度公式为：  
  `sim(u,v) = ∑(i∈N(u)∩N(v)) 1/log(1+|N(i)|) / sqrt(|N(u)|*|N(v)|)`  
  （N(u)表示用户u评分的电影集合，|N(i)|表示电影i的评分用户数）  
  - **效果**：降低热门电影对相似度计算的干扰，提升长尾推荐能力
  - **合理性**：引入流行度惩罚因子是合理的，能减少热门电影对相似度计算的干扰，但需调整实现方式。
  - **问题**：当前使用corr计算皮尔逊相关系数，无法直接集成惩罚因子。需手动实现相似度计算。
---

### **2. 倒排表加速（AI说性能提升明显，但这里反而负优化了，好像得要大规模数据才明显？）**
- 构建电影到用户的倒排索引：  
  ```python
  movie_user_dict = defaultdict(set)
  for user, movie in ratings[['userId', 'movieId']].itertuples(index=False):
      movie_user_dict[movie].add(user)
  ```
  **效果**：将计算复杂度从O(n_users²)降为O(n_movies×avg_users²)

---


### **3. 冷启动策略增强**
- 混合推荐策略：  
  举个栗子：`冷启动推荐 = 60%全局热门 + 30%近期热门（用之前提到的timestamp那一列数据） + 10%随机多样性`  
  **效果**：平衡热度与多样性，避免推荐列表过于陈旧

---

### **4. 加权评分聚合（提升推荐质量 已经实现）**
- 修改推荐得分公式：  
  `电影得分 = ∑(相似用户相似度 × 该用户对此电影的归一化评分)`  
  **效果**：相比简单计数，更能反映用户的真实兴趣强度

---

### **5. 评估指标集成（质量监控）**
- 添加基础离线评估：  
  ```python
  # 计算准确率（需预留测试集）
  precision = len(recommendations & test_set) / top_n
  # 计算覆盖率
  coverage = len(recommended_movies) / total_movies
  ```
    //太麻烦了，所以这里的代码我没有具体实现
---
### 6. **其他参考文献**
[可以参考这篇转载的文章](https://blog.csdn.net/MAGANG255/article/details/52274014?locationNum=15&fps=1)

### 7.修改后的代码（部分修改参考了AI，感觉有些地方改的有点过了）

In [190]:
import pandas as pd
import numpy as np
import time
import math
from collections import defaultdict
from tqdm import tqdm
from sklearn.preprocessing import normalize

# 数据加载与增强
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

# 构建增强的评分矩阵（考虑时间衰减）
current_time = time.time() if 'timestamp' in ratings else 0
ratings['weight'] = 1.0
if 'timestamp' in ratings:
    ratings['days_ago'] = (current_time - ratings['timestamp']) / (3600*24)
    ratings['weight'] = np.exp(-ratings['days_ago']/180)  # 半年衰减因子

ratings_df = ratings.pivot_table(index='userId', columns='movieId', 
                               values='rating', aggfunc='first').fillna(0)
weights_df = ratings.pivot_table(index='userId', columns='movieId',
                               values='weight', aggfunc='first').fillna(0)

# 数据结构优化
print("构建增强数据结构...")
movie_user_dict = defaultdict(set)
user_movie_dict = defaultdict(dict)
user_mean_rating = defaultdict(float)
movie_mean_rating = defaultdict(float)

for user, movie, rating, weight in tqdm(
    ratings[['userId', 'movieId', 'rating', 'weight']].itertuples(index=False),
    total=len(ratings)
):
    movie_user_dict[movie].add(user)
    user_movie_dict[user][movie] = (rating, weight)
    # 更新用户平均评分（加权）
    user_mean_rating[user] = (user_mean_rating.get(user, 0) * sum(w for _,w in user_movie_dict[user].values()) + rating*weight) / sum(w for _,w in user_movie_dict[user].values())
    # 更新电影平均评分
    movie_mean_rating[movie] = (movie_mean_rating.get(movie, 0) * len(movie_user_dict[movie]) + rating) / (len(movie_user_dict[movie]) + 1)

# 电影特征提取
movie_popularity = {m: len(u) for m, u in movie_user_dict.items()}
avg_popularity = np.mean(list(movie_popularity.values())) if movie_popularity else 1
movie_novelty = {m: 1/math.log(1+len(u)) for m, u in movie_user_dict.items()}

# 基于SVD的降维优化
def svd_enhanced_similarity(n_components=50):
    print("计算SVD增强相似度...")
    from sklearn.decomposition import TruncatedSVD
    from sklearn.metrics.pairwise import cosine_similarity
    
    # 归一化评分矩阵
    norm_ratings = ratings_df.subtract(ratings_df.mean(axis=1), axis=0)
    norm_ratings = norm_ratings.div(norm_ratings.std(axis=1), axis=0).fillna(0)
    
    # 使用TruncatedSVD降维
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    svd_matrix = svd.fit_transform(norm_ratings)
    
    # 计算降维后的余弦相似度
    sim_matrix = cosine_similarity(svd_matrix)
    return pd.DataFrame(sim_matrix, index=ratings_df.index, columns=ratings_df.index)

user_similarity = svd_enhanced_similarity()

# 动态候选电影生成
def get_candidate_movies(target_user_id, similar_users, diversity_factor=0.3):
    target_rated = set(user_movie_dict.get(target_user_id, {}).keys())
    candidate_scores = defaultdict(float)
    
    # 第一阶段：基于相似用户推荐
    for sim_user in similar_users:
        sim_score = user_similarity.at[target_user_id, sim_user]
        for movie, (rating, weight) in user_movie_dict.get(sim_user, {}).items():
            if movie not in target_rated:
                # 综合相似度、评分质量和新颖性
                penalty = movie_novelty.get(movie, 1) * (0.5 + 0.5*weight)
                candidate_scores[movie] += sim_score * (rating - user_mean_rating[sim_user]) * penalty
    
    # 第二阶段：加入多样性扰动
    if candidate_scores:
        avg_score = np.mean(list(candidate_scores.values()))
        for m in candidate_scores:
            # 对非热门电影给予额外加分
            if movie_popularity[m] < avg_popularity:
                candidate_scores[m] *= (1 + diversity_factor * (avg_popularity/movie_popularity[m]))
    
    return candidate_scores

# 多阶段冷启动推荐
def get_cold_start_recommendations(top_n=5):
    # 阶段1：高质量热门电影（40%）
    quality_popular = ratings.groupby('movieId').agg(
        mean_rating=('rating','mean'),
        count=('rating','count')
    ).query('count > 10').sort_values(
        by=['mean_rating','count'], ascending=[False,False]
    ).index.tolist()[:2*top_n]
    
    # 阶段2：近期热门（30%）
    recent_popular = []
    if 'timestamp' in ratings:
        try:
            recent_cutoff = ratings['timestamp'].max() - 3600*24*90  # 3个月
            recent_popular = ratings[ratings['timestamp'] > recent_cutoff]\
                .groupby('movieId')['rating'].agg(['mean','count'])\
                .query('count > 5').sort_values(
                    by=['mean','count'], ascending=[False,False]
                ).index.tolist()[:top_n]
        except:
            pass
    
    # 阶段3：类型多样性（20%）
    from collections import Counter
    genre_counter = Counter()
    for genres in movies['genres']:
        genre_counter.update(genres.split('|'))
    
    diverse_movies = []
    for movie in movies['movieId']:
        if movie in movie_popularity and movie_popularity[movie] < 2*avg_popularity:
            genres = movies[movies['movieId']==movie]['genres'].iloc[0].split('|')
            diversity_score = sum(1/genre_counter[g] for g in genres)
            diverse_movies.append((movie, diversity_score))
    diverse_movies = [m for m,_ in sorted(diverse_movies, key=lambda x: x[1], reverse=True)][:top_n]
    
    # 智能混合
    recommendations = []
    sources = [
        (quality_popular, 0.4),
        (recent_popular, 0.3),
        (diverse_movies, 0.2),
        (list(movie_popularity.keys()), 0.1)  # 随机补全
    ]
    
    for source, ratio in sources:
        n = max(1, int(top_n * ratio))
        candidates = [m for m in source if m not in recommendations]
        if candidates:
            selected = candidates[:n]
            if source == list(movie_popularity.keys()):  # 随机选择
                selected = np.random.choice(candidates, size=min(n,len(candidates)), replace=False).tolist()
            recommendations.extend(selected)
    
    return [movie_id_name.get(mid, f"未知电影 (ID: {mid})") for mid in recommendations[:top_n]]

# 最终推荐生成
def generate_recommendations(target_user_id, top_n=5):
    # 冷启动处理
    if target_user_id not in user_id_map:
        return get_cold_start_recommendations(top_n)
    
    # 精确相似用户筛选
    sim_users = user_similarity[target_user_id].sort_values(ascending=False)[1:51]  # 排除自己
    sim_users = sim_users[sim_users > 0.35].index.tolist()  # 更高相似度阈值
    
    if not sim_users:
        return get_cold_start_recommendations(top_n)
    
    # 生成候选电影（带多样性因子）
    candidate_scores = get_candidate_movies(target_user_id, sim_users)
    
    if not candidate_scores:
        return get_cold_start_recommendations(top_n)
    
    # 最终排序（考虑预测评分和多样性）
    sorted_movies = sorted(candidate_scores.items(), 
                          key=lambda x: (x[1], np.random.random()),  # 加入随机扰动
                          reverse=True)
    
    # 结果后处理（防止同系列电影扎堆）
    final_recs = []
    genres_seen = set()
    for mid, score in sorted_movies:
        if len(final_recs) >= top_n:
            break
        genres = set(movies[movies['movieId']==mid]['genres'].iloc[0].split('|'))
        if not genres.intersection(genres_seen) or len(final_recs) >= top_n-1:
            final_recs.append(mid)
            genres_seen.update(genres)
    
    return [movie_id_name.get(mid, f"未知电影 (ID: {mid})") for mid in final_recs[:top_n]]

if __name__ == "__main__":
    start_time = time.time()
    
    target_users = [1, 15, 99999]
    print("用户存在性验证：")
    for uid in target_users:
        exists = "存在" if uid in user_id_map else "不存在"
        print(f"  用户 {uid}: {exists}")
    
    for user_id in target_users:
        print(f"\n=== 用户 {user_id} 的推荐 ===")
        try:
            recs = generate_recommendations(user_id)
            for i, title in enumerate(recs, 1):
                print(f"{i}. {title}")
        except Exception as e:
            print(f"错误：{str(e)}")
    
    end_time = time.time()
    print(f"\n总运行时间: {end_time - start_time:.2f} 秒")

构建增强数据结构...


100%|██████████| 100836/100836 [00:03<00:00, 29829.65it/s]


计算SVD增强相似度...
用户存在性验证：
  用户 1: 存在
  用户 15: 存在
  用户 99999: 不存在

=== 用户 1 的推荐 ===
1. Lives of Others, The (Das leben der Anderen) (2006)
2. Maltese Falcon, The (a.k.a. Dangerous Female) (1931)
3. Corporation, The (2003)
4. Looper (2012)
5. Moon (2009)

=== 用户 15 的推荐 ===
1. American President, The (1995)
2. Executive Decision (1996)
3. Sleeping Beauty (1959)
4. Jacob's Ladder (1990)
5. Secret of NIMH, The (1982)

=== 用户 99999 的推荐 ===
1. Secrets & Lies (1996)
2. Guess Who's Coming to Dinner (1967)
3. Shawshank Redemption, The (1994)
4. La cravate (1957)
5. Doctor Who: Last Christmas (2014)

总运行时间: 1.55 秒


# 5.分析以及经验总结
一些误区以及对错误的分析

### 1. **归一化步骤多余且不合理**
- **分析**：在前几版本的代码中，我 **使用用户平均评分对数据进行归一化** 后再计算皮尔逊相关系数。然而，皮尔逊相关系数在计算时已自动对数据进行中心化（即减去均值）。手动归一化导致重复处理，且可能引发错误（如用户评分全相同导致标准差为零，相似度计算失败）。
- **建议**：移除归一化步骤，直接使用原始评分矩阵计算用户相似度即可。

### 2. **使用 电影平均分或者0值 填充空值NaN**
- **分析**：使用电影平均分填充空值会带来一些潜在问题：
    1. 热门电影会主导推荐结果（评分多的电影均值更可靠）
    2. 人为添加了不存在的评分关联
    3. 如果你不仅手动归一化了还使用了fillna(0)，那么：
        - 在归一化后使用 fillna(0) 填充缺失值，导致未评分的电影被错误地视为“评分为用户平均分”（即归一化后的零值）。
        - 在计算用户相似度时，这些填充的零值会被纳入计算，使得两个用户因大量未评分的电影（填充的零值）而被误判为相似。

- **建议**：直接不管NaN，后面计算皮尔逊相关系数的时候会自动处理的


# Part2-基于模型的协同过滤
---

>试了好久还是没能训练好模型，只能期待后人的智慧了

In [90]:
import time
import numpy as np
import pandas as pd
from collections import defaultdict

# ==================== 数据预处理阶段 ====================
print("正在创建评分矩阵...")
start_time = time.time()

# 读取数据
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

# 创建电影ID到类型的映射
movie_genres = {}
for idx, row in movies.iterrows():
    movie_genres[row['movieId']] = row['genres'].split('|')

# 创建评分矩阵
raw_rating_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')
movie_ids = raw_rating_matrix.columns.values
user_ids = raw_rating_matrix.index.values
print(f"评分矩阵创建完成，耗时：{time.time()-start_time:.2f}秒")

# ==================== 训练测试集划分 ====================
print("\n正在划分训练测试集...")
np.random.seed(42)

# 创建训练矩阵和测试掩码
train_matrix = raw_rating_matrix.copy().values
test_mask = np.zeros_like(train_matrix, dtype=bool)

for u in range(raw_rating_matrix.shape[0]):
    rated = np.where(~np.isnan(train_matrix[u]))[0]
    if len(rated) > 1:
        test_size = max(1, int(len(rated)*0.2))
        test_indices = np.random.choice(rated, test_size, replace=False)
        train_matrix[u, test_indices] = np.nan
        test_mask[u, test_indices] = True

print(f"训练测试集划分完成，耗时：{time.time()-start_time:.2f}秒")

# ==================== 矩阵分解模型 ====================
print("\n开始训练矩阵分解模型...")
start_time = time.time()

# 超参数
n_factors = 100
lr = 0.001
reg = 0.05
epochs = 30

# 初始化参数
n_users, n_items = train_matrix.shape
global_mean = np.nanmean(train_matrix)
user_bias = np.zeros(n_users)
item_bias = np.zeros(n_items)
np.random.seed(42)
user_factors = np.random.normal(0, 0.1, (n_users, n_factors))
item_factors = np.random.normal(0, 0.1, (n_items, n_factors))

# 准备训练数据
train_data = [(u, i, train_matrix[u,i]) 
             for u in range(n_users) 
             for i in range(n_items) 
             if not np.isnan(train_matrix[u,i])]

# 训练循环
for epoch in range(1, epochs+1):
    epoch_start = time.time()
    np.random.shuffle(train_data)
    
    for u, i, r in train_data:
        pred = global_mean + user_bias[u] + item_bias[i] + user_factors[u] @ item_factors[i]
        error = r - pred
        
        # 更新参数
        user_bias[u] += lr * (error - reg*user_bias[u])
        item_bias[i] += lr * (error - reg*item_bias[i])
        
        u_factor = user_factors[u].copy()
        i_factor = item_factors[i].copy()
        
        user_factors[u] += lr * (error*i_factor - reg*u_factor)
        item_factors[i] += lr * (error*u_factor - reg*i_factor)
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch} 完成，耗时：{time.time()-epoch_start:.2f}秒")

print(f"模型训练完成，总耗时：{time.time()-start_time:.2f}秒")

# ==================== 模型评估 ====================
print("\n正在评估模型...")
start_time = time.time()

# 生成预测矩阵
pred_matrix = global_mean + user_bias[:, None] + item_bias[None, :] + user_factors @ item_factors.T

# RMSE和MAE计算
test_ratings = raw_rating_matrix.values[test_mask]
test_preds = pred_matrix[test_mask]
rmse = np.sqrt(np.mean((test_ratings - test_preds)**2))
mae = np.mean(np.abs(test_ratings - test_preds))
print(f"评估结果：RMSE={rmse:.4f}, MAE={mae:.4f}")

# ==================== 高级推荐指标 ====================
# 准备必要数据
THRESHOLD = 4.0  # 评分阈值
TOP_N = 10       # 推荐列表长度

# 获取所有电影ID
all_movie_ids = movie_ids

# 生成流行电影列表（根据评分次数）
movie_popularity = ratings.groupby('movieId').size()
popular_movies = movie_popularity.sort_values(ascending=False).index.tolist()

# 构建用户实际喜欢的电影集合（测试集中评分>=4的）
user_liked_movies = defaultdict(set)
test_user_ids, test_movie_ids = np.where(test_mask)
for u_idx, m_idx in zip(test_user_ids, test_movie_ids):
    if raw_rating_matrix.values[u_idx, m_idx] >= THRESHOLD:
        user_id = user_ids[u_idx]
        movie_id = movie_ids[m_idx]
        user_liked_movies[user_id].add(movie_id)

# 构建训练集已评分电影集合
train_rated_movies = defaultdict(set)
train_user_ids, train_movie_ids = np.where(~np.isnan(train_matrix))
for u_idx, m_idx in zip(train_user_ids, train_movie_ids):
    user_id = user_ids[u_idx]
    movie_id = movie_ids[m_idx]
    train_rated_movies[user_id].add(movie_id)

# 准备类型多样性数据
all_genres = set()
for genres in movie_genres.values():
    all_genres.update(genres)
total_genres = len(all_genres)

# 初始化评估指标
metrics = {
    'precision': 0,
    'recall': 0,
    'coverage': set(),
    'diversity': 0,
    'popularity': 0
}

# 遍历每个用户生成推荐
for user_id in user_ids:
    # 跳过没有喜欢电影的用户
    if user_id not in user_liked_movies or len(user_liked_movies[user_id]) == 0:
        continue
    
    u_idx = np.where(user_ids == user_id)[0][0]
    user_pred = pred_matrix[u_idx]
    
    # 生成候选电影（排除训练集已评分的）
    candidate_mask = [mid not in train_rated_movies[user_id] for mid in movie_ids]
    candidate_ids = movie_ids[candidate_mask]
    candidate_scores = user_pred[candidate_mask]
    
    # 加入流行度降权
    popularity_scores = np.array([1 - (popular_movies.index(mid)/len(popular_movies)) 
                                if mid in popular_movies else 1.0 
                                for mid in candidate_ids])
    combined_scores = 0.7*candidate_scores + 0.3*popularity_scores
    
    # 获取TopN推荐
    top_indices = np.argsort(-combined_scores)[:TOP_N]
    recommendations = candidate_ids[top_indices]
    
    # 计算指标
    liked = user_liked_movies[user_id]
    recommended = set(recommendations)
    
    # Precision and Recall
    tp = len(recommended & liked)
    precision = tp / TOP_N
    recall = tp / len(liked) if len(liked) > 0 else 0
    
    # Coverage
    metrics['coverage'].update(recommendations)
    
    # Diversity
    rec_genres = set()
    for mid in recommendations:
        rec_genres.update(movie_genres.get(mid, []))
    metrics['diversity'] += len(rec_genres) / total_genres
    
    # Popularity
    metrics['popularity'] += np.mean([popular_movies.index(mid)+1 if mid in popular_movies else len(popular_movies) 
                                    for mid in recommendations])
    
    metrics['precision'] += precision
    metrics['recall'] += recall

# 计算平均值
num_eval_users = len(user_liked_movies)
metrics['precision'] /= num_eval_users
metrics['recall'] /= num_eval_users
metrics['diversity'] /= num_eval_users
metrics['coverage'] = len(metrics['coverage']) / len(all_movie_ids)
metrics['popularity'] /= num_eval_users

print(f"平均 Precision@{TOP_N}: {metrics['precision']:.4f}")
print(f"平均 Recall@{TOP_N}: {metrics['recall']:.4f}")
print(f"覆盖率: {metrics['coverage']:.4f}")
print(f"多样性: {metrics['diversity']:.4f}")
print(f"平均流行度排名: {metrics['popularity']:.1f}")
print(f"评估耗时：{time.time()-start_time:.2f}秒")

# ==================== 推荐生成 ====================
print("\n正在生成推荐列表...")

# 预计算用户实际评分过的电影
user_rated_movies = ratings.groupby('userId')['movieId'].apply(set).to_dict()  # 每个用户评分过的电影集合

# 电影元数据映射
movie_id_name = movies.set_index('movieId')['title'].to_dict()  # 电影 ID 到名称的映射
all_movie_ids = raw_rating_matrix.columns.tolist()  # 所有电影 ID
popular_movies = ratings['movieId'].value_counts().index.tolist()  # 按评分次数排序的热门电影列表

# 测试用户列表
test_users = [1, 15, 99999]

# 遍历测试用户，生成推荐
for uid in test_users:
    print(f"\n=== 用户 {uid} 的推荐 ===")
    start = time.time()
    
    try:
        # 冷启动处理
        if uid not in raw_rating_matrix.index:  # 如果用户不存在于评分矩阵中
            recommendations = [movie_id_name.get(mid, f"未知电影 (ID: {mid})") for mid in popular_movies[:10]]  # 返回热门电影
        else:
            # 获取用户实际评分过的电影
            rated = user_rated_movies.get(uid, set())  # 用户已评分的电影集合
            
            # 生成候选电影列表
            user_idx = raw_rating_matrix.index.get_loc(uid)  # 获取用户在评分矩阵中的索引
            user_preds = pred_matrix[user_idx]  # 用户对所有电影的预测评分
            
            candidates = []
            for mid, score in zip(all_movie_ids, user_preds):  # 遍历所有电影
                if mid not in rated:  # 跳过用户已评分的电影
                    popularity = popular_movies.index(mid)/len(popular_movies) if mid in popular_movies else 1.0  # 计算电影流行度
                    combined_score = 0.7*score + 0.3*(1 - popularity)  # 综合评分（预测评分 + 流行度）
                    candidates.append((mid, combined_score))  # 添加候选电影
            
            # 按得分排序
            candidates.sort(key=lambda x: x[1], reverse=True)
            recommendations = [movie_id_name.get(mid, f"未知电影 (ID: {mid})") for mid, _ in candidates[:10]]  # 返回前 10 个推荐
        
        # 打印推荐结果
        for i, title in enumerate(recommendations, 1):
            print(f"{i}. {title}")
        print(f"耗时：{time.time()-start:.4f}秒")
    
    except Exception as e:
        print(f"生成推荐时出错：{str(e)}")

正在创建评分矩阵...
评分矩阵创建完成，耗时：0.30秒

正在划分训练测试集...
训练测试集划分完成，耗时：0.35秒

开始训练矩阵分解模型...
Epoch 5 完成，耗时：0.79秒
Epoch 10 完成，耗时：0.95秒
Epoch 15 完成，耗时：0.81秒
Epoch 20 完成，耗时：0.93秒
Epoch 25 完成，耗时：0.87秒
Epoch 30 完成，耗时：0.88秒
模型训练完成，总耗时：30.12秒

正在评估模型...
评估结果：RMSE=0.8984, MAE=0.6924
平均 Precision@10: 0.0843
平均 Recall@10: 0.0586
覆盖率: 0.0104
多样性: 0.5204
平均流行度排名: 68.7
评估耗时：1439.19秒

正在生成推荐列表...

=== 用户 1 的推荐 ===
1. Shawshank Redemption, The (1994)
2. Godfather, The (1972)
3. Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
4. Dark Knight, The (2008)
5. One Flew Over the Cuckoo's Nest (1975)
6. Godfather: Part II, The (1974)
7. Blade Runner (1982)
8. Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
9. Casablanca (1942)
10. Snatch (2000)
耗时：0.7892秒

=== 用户 15 的推荐 ===
1. Casablanca (1942)
2. Fargo (1996)
3. Eternal Sunshine of the Spotless Mind (2004)
4. Braveheart (1995)
5. Reservoir Dogs (1992)
6. Silence of the Lambs, The (1991)
7. Dr. Strangelove or: How I Learned to Stop Worrying

# Part3-关于ItemCF（在电影推荐这一方面好像UserCF比较适合）
---

## 1. 理论基础
### 1.1 核心思想
基于物品的协同过滤（ItemCF）通过分析用户历史行为数据，计算物品之间的相似度矩阵，为用户推荐与他们历史偏好物品相似的物品。其核心假设是：**用户倾向于喜欢与过去偏好物品相似的物品**

### 1.2 算法流程
1. **构建用户-物品交互矩阵**：记录每个用户的交互物品列表
2. **计算物品相似度**：通过共现分析计算物品间的相似度
3. **生成推荐列表**：根据用户历史物品的相似物品进行加权推荐
4. **评估优化**：通过离线指标评估推荐效果

### 1.3 与UserCF对比
| 维度        | UserCF                         | ItemCF                         |
|------------|-------------------------------|-------------------------------|
| 适用场景    | 用户兴趣变化快                 | 物品库相对稳定                 |
| 推荐解释性  | 较难解释（基于相似用户）       | 易于解释（基于相似物品）       |
| 实时性      | 用户新增行为需重新计算相似度   | 物品相似度矩阵可预先计算       |
| 冷启动      | 新用户无法推荐                 | 新物品难被推荐                 |

### **1.4 确定使用UserCF或ItemCF可参考以下因素**：
1. **数据特点**：数据稀疏选ItemCF；用户多物品少选ItemCF，反之可选UserCF。
2. **推荐场景**：强调个性化选UserCF；挖掘热门趋势、实时性低选ItemCF。
3. **用户行为**：兴趣稳定选ItemCF；兴趣多变选UserCF；用户熟悉物品多可用UserCF，反之选ItemCF辅助探索。

实际应用中，也可将两者结合。 

### 1.5 参考文献
1. [Item-based Collaborative Filtering Recommendation Algorithms](https://dl.acm.org/doi/10.1145/371920.372071) 

## 2.数据预处理
### 2.1导入所需数据和库

In [31]:
import pandas as pd
import numpy as np

# 加载数据
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

# 检查缺失值和重复值
print("重复评分数量:", ratings.duplicated(subset=['userId', 'movieId']).sum())

重复评分数量: 0


## 3.构建ItemCF推荐系统
### 3.1创建物品-用户倒排表

In [33]:
# 创建物品-用户倒排表
item_user_dict = {}
for movie_id, group in ratings.groupby('movieId'):
    item_user_dict[movie_id] = set(group['userId'])

# 物品流行度（被多少用户评分过）
item_popularity = {movie_id: len(users) for movie_id, users in item_user_dict.items()}

### 3.2计算物品相似度
使用余弦相似度计算物品之间的相似性

In [35]:
from collections import defaultdict
import math

# 计算物品共现矩阵
cooccur = defaultdict(lambda: defaultdict(int))
for user, group in ratings.groupby('userId'):
    items = group['movieId'].tolist()
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            cooccur[items[i]][items[j]] += 1
            cooccur[items[j]][items[i]] += 1

# 计算余弦相似度
item_sim_matrix = defaultdict(dict)
for item1 in cooccur:
    for item2 in cooccur[item1]:
        # 余弦相似度公式
        sim = cooccur[item1][item2] / math.sqrt(item_popularity[item1] * item_popularity[item2])
        item_sim_matrix[item1][item2] = sim

### 3.3生成推荐
根据用户历史行为，推荐相似度高的物品

In [37]:
def recommend_items(user_id, top_n=5):
    """基于物品的推荐"""
    # 获取用户评分过的电影
    user_rated = set(ratings[ratings['userId'] == user_id]['movieId'])
    
    # 候选物品及其得分
    item_scores = defaultdict(float)
    for rated_item in user_rated:
        for sim_item, sim_score in item_sim_matrix.get(rated_item, {}).items():
            if sim_item not in user_rated:
                item_scores[sim_item] += sim_score
    
    # 按得分排序
    recommended_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    # 转换为电影名称
    movie_id_name = movies.set_index('movieId')['title'].to_dict()
    return [(movie_id_name.get(mid, f"未知电影 (ID: {mid})"), score) for mid, score in recommended_items]

### 3.4测试推荐

In [39]:
# 测试推荐
user_id = 1
print(f"用户 {user_id} 的推荐列表：")
for movie, score in recommend_items(user_id):
    print(f"- {movie} (相似度：{score:.2f})")

用户 1 的推荐列表：
- Ferris Bueller's Day Off (1986) (相似度：83.42)
- Die Hard (1988) (相似度：81.06)
- Mars Attacks! (1996) (相似度：80.62)
- Back to the Future Part II (1989) (相似度：79.93)
- Breakfast Club, The (1985) (相似度：79.23)


## 4. 优化与改进  

### 4.1 相似度矩阵截断  
仅保留每个物品最相似的 \( k \) 个物品，减少计算量。  

```python
k = 20  # 保留最相似的20个物品
for item in item_sim_matrix:
    item_sim_matrix[item] = dict(sorted(item_sim_matrix[item].items(), key=lambda x: x[1], reverse=True)[:k])
```

### 4.2 流行度惩罚  
降低热门物品的权重，避免推荐过于流行的物品。  

```python
alpha = 0.7  # 惩罚系数
for item1 in item_sim_matrix:
    for item2 in item_sim_matrix[item1]:
        item_sim_matrix[item1][item2] /= (item_popularity[item2] ** alpha)
```

### 4.3 冷启动处理  
如果用户没有历史行为，推荐全局热门物品。  

```python
def recommend_items(user_id, top_n=5):
    user_rated = set(ratings[ratings['userId'] == user_id]['movieId'])
    if not user_rated:
        # 冷启动：推荐热门电影
        popular_movies = ratings['movieId'].value_counts().index[:top_n]
        movie_id_name = movies.set_index('movieId')['title'].to_dict()
        return [(movie_id_name.get(mid, f"未知电影 (ID: {mid})"), 1.0) for mid in popular_movies]
    
    # 正常推荐逻辑...
```

---

## 5. 评估与总结  

### 5.1 评估指标  
（与UserCF部分相同，可复用MAE/RMSE或准确率/召回率）  

### 5.2 经验总结  
1. **ItemCF更适合物品稳定的场景**，如电影、书籍推荐。但是  
2. **相似度矩阵可离线计算**，适合大规模系统。  
3. **热门物品需要惩罚**，否则推荐结果会偏向流行物品。  
4. **冷启动问题**可通过全局热门物品缓解。  


# Part4-最后，那我问你，有没有现成的库直接调用呢？
**有的兄弟，有的，这样的库我们还有四个……**
---

## 常用推荐系统库

1. **Surprise**
   - 专注于推荐系统的Python库
   - 支持UserCF和ItemCF

2. **LightFM**
   - 支持混合协同过滤和内容推荐
   - 安装: `pip install lightfm`

3. **implicit**
   - 专注于隐式反馈的推荐系统
   - 支持ItemCF风格的算法

4. **Cornac**
   - 多模式推荐系统库
   - 支持多种协同过滤算法
 
## 选择建议

- 对于快速实现和评估UserCF/ItemCF，**Surprise**是最直接的选择
- 如果需要处理隐式反馈(如点击、浏览)，**implicit**更合适
- 对于研究或需要更多算法选择，**Cornac**提供了丰富的模型

这些库都提供了评估工具，可以方便地计算准确率、召回率等指标。